In [1]:
%pip install langchain langgraph transformers accelerate sentence-transformers faiss-cpu langchain-huggingface

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB 330.3 kB/s eta 0:00:58
   ---------------------------------------- 0.1/18.9 MB 787.7 kB/s eta 0:00:24
   - -------------------------------------- 0.8/18.9 MB 6.4 MB/s eta 0:00:03
   -- ------------------------------------- 1.0/18.9 MB 6.6 MB/s eta 0:00:03
   --- ------------------------------------ 1.7/18.9 MB 8.4 MB/s eta 0:00:03
   ----- ---------------------------------- 2.6/18.9 MB 9.9 MB/s eta 0:00:02
   ------- -------------------------------- 3.6/18.9 MB 11.7 MB/s eta 0:00:02
   --------- ------------------------------ 4.6/18.9 MB 12.8 MB/s eta 0:00:02
   ----------- ---------------------------- 5.3/18.9 MB 12.9 MB/s eta 0:00:02
   ------------ --------------------------- 6.1/18.9 MB 13.5 MB/s eta 0:00:01
   ------------- -------------------------- 6.4/18.9 MB 13.2 MB/s eta 0:00:01
   -------------- ------------------------- 7.0/18.9 MB 12.8 MB/s eta 0:00


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [3]:
pipe = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=50
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

print("Modelo cargado correctamente")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo cargado correctamente


In [4]:
from langchain_core.tools import Tool

In [5]:
guest_data = {
    "marie curie": "Marie Curie was a physicist and chemist known for radioactivity.",
    "leonardo da vinci": "Leonardo da Vinci was a Renaissance artist and inventor.",
    "isaac newton": "Isaac Newton developed the laws of motion and gravity."
}

In [6]:
def search_guest(name: str):
    
    name = name.lower()
    
    if name in guest_data:
        return guest_data[name]
    
    return "Guest not found."

In [7]:
guest_tool = Tool(
    name="guest_search",
    func=search_guest,
    description="Searches information about gala guests."
)

In [8]:
print(
    guest_tool.invoke("marie curie")
)

Marie Curie was a physicist and chemist known for radioactivity.


In [9]:
def weather_tool(city: str):
    
    return f"The weather in {city} is clear for the fireworks."

In [10]:
weather = Tool(
    name="weather_tool",
    func=weather_tool,
    description="Provides weather information."
)

In [11]:
print(
    weather.invoke("Paris")
)

The weather in Paris is clear for the fireworks.


In [12]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

In [13]:
class AgentState(TypedDict):
    question: str
    answer: str

In [14]:
def agent_node(state):
    
    question = state["question"].lower()
    
    if "marie curie" in question:
        
        answer = guest_tool.invoke("marie curie")
    
    elif "weather" in question:
        
        answer = weather.invoke("Paris")
    
    else:
        
        answer = "I do not have information about that."
    
    return {
        "answer": answer
    }

In [15]:
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)

builder.add_edge(START, "agent")
builder.add_edge("agent", END)

graph = builder.compile()

In [16]:
result = graph.invoke({
    "question": "Tell me about Marie Curie",
    "answer": ""
})

print(result)

{'question': 'Tell me about Marie Curie', 'answer': 'Marie Curie was a physicist and chemist known for radioactivity.'}


In [17]:
result = graph.invoke({
    "question": "What is the weather today?",
    "answer": ""
})

print(result)

{'question': 'What is the weather today?', 'answer': 'The weather in Paris is clear for the fireworks.'}
